# Demo 05: Steering

A guardrail can say **no** (hard block → the task fails) or **here's how** (steering → the agent fixes the request and finishes). Rule here: a room holds at most **10 guests**. The agent books the whole party in one call; the guardrail — which knows the real limit — reacts. Ask to book **15** and watch each approach.

**Prerequisites:** Neo4j with the demo-01 hotel graph; Amazon Bedrock access.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json, os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

from dotenv import load_dotenv
from strands import Agent

from booking_store import BookingStore  # JSON store; book_hotel has a `guests` param
from steering_tools import HotelCheck    # read-only Neo4j hotel lookup

load_dotenv()

# --- Model: Amazon Bedrock by default ---
# The agents below are created with Agent(...) and no model=, so Strands uses
# its default Bedrock provider automatically (nothing to import).
# To use OpenAI instead: pip install 'strands-agents[openai]'
# (also listed, commented, in requirements.txt), set OPENAI_API_KEY in .env,
# create the model, and pass it to each Agent(...) via model=OPENAI_MODEL:
# from strands.models.openai import OpenAIModel
# OPENAI_MODEL = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id="gpt-4o-mini")

hotels = HotelCheck()
PROMPT = (
    "You are a travel agent that helps people with hotels. Book the whole party "
    "together with the full guest count; do not assume any per-room limit yourself. "
    "If you receive guidance to split the booking, follow it exactly. Keep answers concise. "
    ""
)
REQUEST = "Book 15 guests at Cliffside Resort for Jordan Lee."

def fresh(path):  # start each run from an empty store
    if os.path.exists(path): os.remove(path)
    return BookingStore(path)

---
## Part 1: Blocking hook fails the task

A lifecycle hook cancels `book_hotel` when `guests > 10`. The agent asks for the whole party in one call, the hook rejects it, and the booking never happens.

In [ ]:
from strands.hooks import BeforeToolCallEvent, HookProvider, HookRegistry

MAX_PER_ROOM = 10

class BlockOversizeRoomHook(HookProvider):
    """Cancel any book_hotel call over MAX_PER_ROOM guests."""
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeToolCallEvent, self._block)

    def _block(self, event: BeforeToolCallEvent) -> None:
        tu = event.tool_use
        if tu.get("name") != "book_hotel":
            return
        guests = (tu.get("input") or {}).get("guests", 1)
        if guests and guests > MAX_PER_ROOM:
            print(f"[HOOK] ⛔ Blocked: {guests} guests exceeds {MAX_PER_ROOM}/room")
            event.cancel_tool = f"Rejected: {guests} guests exceeds the {MAX_PER_ROOM}-per-room limit."

In [ ]:
# Run 1 — blocking hook: the 15-guest call is cancelled, nothing is booked
blocked_store = fresh("bookings_blocked.json")
agent = Agent(
    system_prompt=PROMPT,
    tools=[hotels.hotel_exists, blocked_store.book_hotel],
    hooks=[BlockOversizeRoomHook()],
)
result = agent(REQUEST)

booked = json.load(open("bookings_blocked.json")) if os.path.exists("bookings_blocked.json") else {}
print(f"\nrooms booked: {len(booked)} — the hook only says NO; any recovery is the model guessing, unguided")

The hook stops the harm but leaves the user stuck. (Same hook mechanism as demo 04 — not re-run here.)

---
## Part 2: Steering completes the task

A native Strands `SteeringHandler` reads the **real** tool input (`tool_use["input"]["guests"]`). When it is too big it returns a `Guide` telling the agent to split into `ceil(guests/10)` rooms and book each. No regex on narrated text, no server.

In [ ]:
import math
from strands.vended_plugins.steering import (
    SteeringHandler, Guide, Proceed, ToolSteeringAction,
)

def split(guests, cap=MAX_PER_ROOM):
    """Full rooms of `cap`, plus a remainder: 15 -> [10, 5], 23 -> [10, 10, 3]."""
    full, rem = divmod(guests, cap)
    return [cap] * full + ([rem] if rem else [])

class MaxGuestsSteering(SteeringHandler):
    name = "max-guests"
    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "book_hotel":
            return Proceed()
        guests = (tool_use.get("input") or {}).get("guests", 1)
        if guests and guests > MAX_PER_ROOM:
            rooms = split(guests)
            print(f"[STEERING] 🧭 {guests} > {MAX_PER_ROOM}/room → split into {rooms}")
            return Guide(reason=(
                f"A room holds at most {MAX_PER_ROOM} guests, but you requested {guests}. "
                f"Split the party into {len(rooms)} rooms of {rooms} guests and call "
                f"book_hotel once per room."))
        return Proceed()

print("split(15) =", split(15), "| split(25) =", split(25))  # generic, not hardcoded

In [ ]:
# Run 2 — native steering: the 15-guest call is guided into 10 + 5, both booked
store = fresh("bookings.json")
agent = Agent(
    system_prompt=PROMPT,
    tools=[hotels.hotel_exists, store.book_hotel],
    plugins=[MaxGuestsSteering()],  # SteeringHandler is a Strands plugin
)
result = agent(REQUEST)

bookings = json.load(open("bookings.json"))
print(f"\n✅ {len(bookings)} rooms booked:")
for bid, b in bookings.items():
    print(f"  {bid}: {b['guests']} guests at {b['hotel']}")
print("total guests:", sum(b["guests"] for b in bookings.values()))

---
## Part 3: Rule from config (optional)

Same native steering, but the limit is loaded from `rules.json`. Rules can live outside the code — no Agent Control server needed. (If you use Agent Control, it is strictly optional; this demo runs fully without it.)

In [ ]:
rules = json.load(open("rules.json"))  # {"max_guests_per_room": 10}

class RuleSteering(SteeringHandler):
    name = "rule-steering"
    def __init__(self, rules):
        super().__init__()
        self._cap = int(rules["max_guests_per_room"])
    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "book_hotel":
            return Proceed()
        guests = (tool_use.get("input") or {}).get("guests", 1)
        if guests and guests > self._cap:
            rooms = split(guests, self._cap)
            return Guide(reason=(
                f"Policy (from config): max {self._cap} per room. Split {guests} into "
                f"{len(rooms)} rooms of {rooms} and book each."))
        return Proceed()

print("✅ RuleSteering loaded cap from config:", rules["max_guests_per_room"])

---
## 🎯 Try It Yourself

Change the party size — the split is generic (`ceil(n/10)`), not hardcoded.

In [ ]:
# A bigger party → more rooms, still completes
try_store = fresh("bookings_try.json")
agent = Agent(
    system_prompt=PROMPT,
    tools=[hotels.hotel_exists, try_store.book_hotel],
    plugins=[MaxGuestsSteering()],
)
result = agent("Book 23 guests at Bay View Grand Hotel for Sam Rivera.")

print("\nrooms:", [b["guests"] for b in json.load(open("bookings_try.json")).values()])

---
## What's Next

These five defenses run locally. In **[Demo 06 — Amazon Bedrock AgentCore](../06-agentcore-boto3-demo/)** you take the same patterns to production with AgentCore Runtime, Lambda, and DynamoDB.